# Render every Phase-5 blade — INTERACTIVE, grouped PASSED vs FAILED

**Fresh Colab session — reads your Drive, re-runs no phase.** Rebuilds **every** verified blade's
3D CAD solid from the saved design numbers and draws it with **Plotly** — so each blade is
**draggable**: click-drag to rotate, scroll to zoom, right-drag to pan. Shown in two groups:
**valid** (green) and **suspect / failed** (red). Uses true proportions, so the blades look like
the long, thin panels they are (not squished cubes). CadQuery + Plotly only — no gmsh/SU2.

## 1. Repo + deps + mount Drive

In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    REPO = Path("/content/fan-optimization")
    if not REPO.exists():
        subprocess.run(["git","clone","-b","main","https://github.com/clingergab/fan-optimization.git",str(REPO)], check=True)
    else:
        subprocess.run(["git","-C",str(REPO),"pull","origin","main"], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","cadquery","plotly"], check=True)
    from google.colab import drive; drive.mount("/content/drive")
else:
    REPO = Path.cwd()
    while REPO != REPO.parent and not (REPO/"pyproject.toml").exists(): REPO = REPO.parent
if str(REPO/"src") not in sys.path: sys.path.insert(0, str(REPO/"src"))
print("repo:", REPO, "| colab:", IN_COLAB)

## 2. Load the data + split passed vs failed

`checkpoint.npz` → every design's 35 numbers (`x`). `verification.json`'s `ranking.pairs` flags
each verified design: `suspect=False` → **passed**, `suspect=True` → **failed / suspect**.

In [ ]:
import numpy as np, json
CAMPAIGN = Path("/content/drive/MyDrive/fanopt/phase4_bo") if IN_COLAB else REPO/"data"/"phase4_bo_nb"
VERIFY   = Path("/content/drive/MyDrive/fanopt/phase5_verify") if IN_COLAB else REPO/"data"/"phase5_verify_nb"

x = np.load(CAMPAIGN/"checkpoint.npz")["x"]
pairs = json.loads((VERIFY/"verification.json").read_text())["ranking"]["pairs"]
passed = [p for p in pairs if not p["suspect"]]
failed = [p for p in pairs if p["suspect"]]
print(f"{len(pairs)} verified total  ->  {len(passed)} passed, {len(failed)} failed/suspect")

## 3. Build + draw helpers (Plotly = draggable)

`design numbers -> decode -> make_vunit_blade` (CAD solid) `-> tessellate` (triangles). Plotly's
`Mesh3d` takes the triangle vertices (`x,y,z`) and the triangle corner-indices (`i,j,k`).
`aspectmode="data"` keeps the real proportions.

In [ ]:
import plotly.graph_objects as go
from fanopt.bo.codec import decode
from fanopt.geometry.generator import BladeDesignParams
from fanopt.geometry.fields import Layer2Params
from fanopt.geometry.primitives import Layer3Primitive
from fanopt.bo.inertia import NEUTRAL_LAYER4
from fanopt.geometry.assembly_cad import make_vunit_blade

def blade_figure(vec, name, j3d, color, tol=0.001):
    params = BladeDesignParams(layer1=decode(vec), layer2=Layer2Params.all_inactive(),
                               layer3=Layer3Primitive.absent(), layer4=NEUTRAL_LAYER4)
    verts, tris = make_vunit_blade(params).val().tessellate(tol)
    V = np.array([[p.x, p.y, p.z] for p in verts])
    mesh = go.Mesh3d(x=V[:,0], y=V[:,1], z=V[:,2],
                     i=[t[0] for t in tris], j=[t[1] for t in tris], k=[t[2] for t in tris],
                     color=color, opacity=0.6, flatshading=True)
    title = f"{name}   J_fan_3D={j3d:.2e}" if j3d is not None else f"{name}   (3D failed)"
    fig = go.Figure(data=[mesh])
    fig.update_layout(title=title, height=520, margin=dict(l=0, r=0, t=40, b=0),
                      scene=dict(aspectmode="data"))
    return fig

def show_group(group, color, heading):
    print("=" * 60); print(heading); print("=" * 60)
    for p in group:
        idx = int(p["name"].split("_i")[-1])
        try:
            blade_figure(x[idx], p["name"], p.get("j_fan_3d"), color).show()
        except Exception as e:
            print(f"  {p['name']}: CAD build failed - {type(e).__name__}: {e}")

## 4. PASSED designs — clean, buildable (drag to rotate; these are your real candidates)

In [ ]:
show_group(passed, "seagreen", f"PASSED ({len(passed)}) — valid, 3D-verified")

## 5. FAILED / SUSPECT — self-intersecting or reverse-thrust (drag to see the fold-through)

In [ ]:
show_group(failed, "crimson", f"FAILED / SUSPECT ({len(failed)}) — rejected")

## 6. Takeaway

Rotate the reds and you'll *see* a surface passing through the body — the `overlapping facets`
gmsh rejected. The greens are clean sheets. The optimizer chased some reds because the cheap 2D
slice couldn't see the 3D fold — exactly what the **V1.5 geometry-validity filter**
(`docs/V2_backlog.md`) would catch up front. Tip: to compare a good vs bad one side by side, drag
each to the same angle.